In [ ]:
!pip install -q datasets


In [ ]:
!pip install -q transformers

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:

import os
os.chdir("/content/gdrive/MyDrive/LLM/CBLLMMODEL")


# Prepare DataSets

In [ ]:
import pandas as pd

df_majority = pd.read_excel("TCMB_ALL_MODELS_MAJORITY.xlsx", index_col="No")
neutral_df = df_majority[df_majority["majority_vote"] == "neutral"].sample(n=2400, random_state=42)
hawkish_df = df_majority[df_majority["majority_vote"] == "hawkish"].sample(n=2400, random_state=42)
dovish_df  = df_majority[df_majority["majority_vote"] == "dovish"].sample(n=2400, random_state=42)
balanced_df = pd.concat([neutral_df, hawkish_df, dovish_df]).sample(frac=1, random_state=42).reset_index(drop=True)
label_map = {"neutral": 0, "hawkish": 1, "dovish": 2}
balanced_df["label"] = balanced_df["majority_vote"].map(label_map)
final_df = balanced_df[["sentence", "label"]].rename(columns={"sentence": "text"})
final_df.to_csv("BALANCED_DATASET.csv", index=False)
final_df.head(3)


# PREPARE DATASET

In [ ]:
from datasets import Dataset
df = pd.read_csv("BALANCED_DATASET.csv") # 🔹 CSV'den veri yükle (text, label sütunları olmalı)
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"} # 🔹 label: neutral=0, hawkish=1, dovish=2
id2label = {v: k for k, v in label_map.items()}
dataset = Dataset.from_pandas(df) # 🔹 Hugging Face Dataset'e çevir
dataset = dataset.train_test_split(test_size=0.2) # 🔹 Train-test split
print(dataset)

# ROBERTA

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset

# 🔹 Tokenizer & Model
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# 🔹 Tokenization fonksiyonu
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

# 🔹 Tokenize işlemi (önceden Hugging Face Dataset formatına çevrilmiş olmalı)
tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 Tokenize
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 Değerlendirme Fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 Eğitim ayarları
training_args = TrainingArguments(
    output_dir="./roberta_hawkish_dovish",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    num_train_epochs=3,
    learning_rate=4.130804819407263e-05,
    weight_decay=0.16782442691636537,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 🔹 Trainer objesi
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🔹 Eğitimi başlat
trainer.train()

# 🔹 Modeli ve tokenizer'ı kaydet
trainer.save_model("MODELS_II/roberta_hawkish_dovish_model")
tokenizer.save_pretrained("MODELS_II/roberta_hawkish_dovish_model")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# 🔹 Tahminleri al
preds = trainer.predict(tokenized["test"])
y_true = np.array(tokenized["test"]["label"])
y_pred = np.argmax(preds.predictions, axis=1)

# 🔹 Sınıf isimleri
label_names = ["neutral", "hawkish", "dovish"]

# 🔹 Genel sınıflandırma raporu
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

# 🔹 Genel accuracy
overall_acc = accuracy_score(y_true, y_pred)
print(f"✅ Overall Accuracy: {overall_acc:.4f}")

# 🔹 Her sınıf için accuracy (class-level)
for label_id, label_name in enumerate(label_names):
    class_mask = (y_true == label_id)
    class_acc = accuracy_score(y_true[class_mask], y_pred[class_mask])
    print(f"🔸 Accuracy for class '{label_name}': {class_acc:.4f}")

In [ ]:

from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

preds = trainer.predict(tokenized["test"])
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_true, y_pred, target_names=["neutral", "hawkish", "dovish"]))
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
acc = accuracy_score(y_true, y_pred)
print(f"✅ Accuracy: {acc:.4f}")

In [ ]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np
# Model klasöründen yükle
model_path = "./roberta_hawkish_dovish_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

In [ ]:
id2label = {0: "neutral", 1: "hawkish", 2: "dovish"}

In [ ]:
def predict_sentence(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.nn.functional.softmax(logits, dim=-1)
        predicted_class = torch.argmax(probs, dim=1).item()
        return {
            "text": text,
            "prediction": id2label[predicted_class],
            "probabilities": dict(zip(id2label.values(), probs[0].tolist()))
        }


In [ ]:
example = """The Monetary Policy Committee has concluded that the data announced
in the recent period since the last meeting did not lead to a significant change
in the inflation and monetary policy outlook.
"""

result = predict_sentence(example)
print("Metin:", result["text"])
print("Tahmin:", result["prediction"])
print("Olasılıklar:", result["probabilities"])

In [ ]:
from transformers import pipeline

# Load the classifier pipeline
classifier = pipeline("text-classification", model="mrince/CBRT-RoBERTa-HawkishDovish-Classifier")

# Example sentence from a monetary policy context
sentence = "On the other hand, the recent deceleration in economic activity may curb services inflation."

# Perform classification
result = classifier(sentence)
print(result)

# Output example:
# [{'label': 'hawkish', 'score': 0.91}]

# Label meanings:
# - 'hawkish [Label_1]': Tightening bias, upward rate signal. Indicates prioritization of inflation control.
# - 'dovish [Label_2]' : Easing bias, downward/inflation-tolerant tone. Indicates support for growth/stimulus.
# - 'neutral [Label_2]': Informational or balanced tone without a clear policy stance.


In [ ]:
# Example sentence from a monetary policy context
sentence = "On the other hand, the recent deceleration in economic activity may curb services inflation.."

# Perform classification
result = classifier(sentence)
print(result[0]["label"])

# ROBERTA LARGE

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 🔹 CSV'den veri yükle
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 label mapping
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 Hugging Face Dataset'e çevir
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 Tokenizer & Model (GÜNCELLENEN KISIM)
model_name = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 Tokenization
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 Değerlendirme metrikleri
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 Eğitim ayarları (BATCH 8 olarak düşürüldü)
training_args = TrainingArguments(
    output_dir="./roberta_large_hawkish_dovish",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=32,
    num_train_epochs=5,
    learning_rate=1.375013558415515e-05,
    weight_decay=0.10819137131766952,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)



# 🔹 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🔹 Eğitimi başlat
trainer.train()

# 🔹 Modeli kaydet
trainer.save_model("MODELS_II/roberta_large_hawkish_dovish_model")
tokenizer.save_pretrained("MODELS_II/roberta_large_hawkish_dovish_model")

In [ ]:

from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# 🔹 Tahminleri al
preds = trainer.predict(tokenized["test"])
y_true = np.array(tokenized["test"]["label"])
y_pred = np.argmax(preds.predictions, axis=1)

# 🔹 Sınıf isimleri
label_names = ["neutral", "hawkish", "dovish"]

# 🔹 Genel sınıflandırma raporu
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

# 🔹 Genel accuracy
overall_acc = accuracy_score(y_true, y_pred)
print(f"✅ Overall Accuracy: {overall_acc:.4f}")

# 🔹 Her sınıf için accuracy (class-level)
for label_id, label_name in enumerate(label_names):
    class_mask = (y_true == label_id)
    class_acc = accuracy_score(y_true[class_mask], y_pred[class_mask])
    print(f"🔸 Accuracy for class '{label_name}': {class_acc:.4f}")


# DİSTİLBERT

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 🔹 CSV'den veri yükle
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 label mapping
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 HF Dataset'e çevir ve train-test ayır
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 Model & Tokenizer (DISTILBERT kullanılıyor)
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 Tokenization
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 Değerlendirme fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 Eğitim ayarları
training_args = TrainingArguments(
    output_dir="./distilbert_hawkish_dovish",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=32,
    num_train_epochs=5,
    learning_rate=4.3545613214606654e-05,
    weight_decay=0.1618129428290227,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 🔹 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🔹 Eğitimi başlat
trainer.train()

# 🔹 Modeli ve tokenizer'ı kaydet
trainer.save_model("MODELS_II/distilbert_hawkish_dovish_model")
tokenizer.save_pretrained("MODELS_II/distilbert_hawkish_dovish_model")

In [ ]:

from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# 🔹 Tahminleri al
preds = trainer.predict(tokenized["test"])
y_true = np.array(tokenized["test"]["label"])
y_pred = np.argmax(preds.predictions, axis=1)

# 🔹 Sınıf isimleri
label_names = ["neutral", "hawkish", "dovish"]

# 🔹 Genel sınıflandırma raporu
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

# 🔹 Genel accuracy
overall_acc = accuracy_score(y_true, y_pred)
print(f"✅ Overall Accuracy: {overall_acc:.4f}")

# 🔹 Her sınıf için accuracy (class-level)
for label_id, label_name in enumerate(label_names):
    class_mask = (y_true == label_id)
    class_acc = accuracy_score(y_true[class_mask], y_pred[class_mask])
    print(f"🔸 Accuracy for class '{label_name}': {class_acc:.4f}")


# DEBERTA V3

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 🔹 CSV'den veri yükle
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 label mapping
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 HF Dataset'e çevir
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 DeBERTa Tokenizer & Model
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 Tokenization
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 Değerlendirme fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 Eğitim ayarları
training_args = TrainingArguments(
    output_dir="./deberta_hawkish_dovish",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    num_train_epochs=6,
    learning_rate=2.15511808238697e-05,
    weight_decay=0.11167437883527277,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 🔹 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🔹 Eğitimi başlat
trainer.train()

# 🔹 Modeli kaydet
trainer.save_model("MODELS_II/deberta_hawkish_dovish_model")
tokenizer.save_pretrained("MODELS_II/deberta_hawkish_dovish_model")


In [ ]:

from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# 🔹 Tahminleri al
preds = trainer.predict(tokenized["test"])
y_true = np.array(tokenized["test"]["label"])
y_pred = np.argmax(preds.predictions, axis=1)

# 🔹 Sınıf isimleri
label_names = ["neutral", "hawkish", "dovish"]

# 🔹 Genel sınıflandırma raporu
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

# 🔹 Genel accuracy
overall_acc = accuracy_score(y_true, y_pred)
print(f"✅ Overall Accuracy: {overall_acc:.4f}")

# 🔹 Her sınıf için accuracy (class-level)
for label_id, label_name in enumerate(label_names):
    class_mask = (y_true == label_id)
    class_acc = accuracy_score(y_true[class_mask], y_pred[class_mask])
    print(f"🔸 Accuracy for class '{label_name}': {class_acc:.4f}")

# Electra

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 🔹 1. Veri Yükleme
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 2. Label Mapping
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 3. HF Dataset ve Train-Test Ayırma
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 4. ELECTRA Tokenizer & Model
model_name = "google/electra-base-discriminator"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 5. Tokenization
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 6. Değerlendirme Fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 7. Eğitim Ayarları
training_args = TrainingArguments(
    output_dir="./electra_hawkish_dovish_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=32,
    num_train_epochs=6,
    learning_rate=3.524362859523875e-05,
    weight_decay=0.16939454533072063,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 🔹 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🔹 9. Eğitimi Başlat
trainer.train()

# 🔹 10. Modeli Kaydet
trainer.save_model("MODELS_II/electra_hawkish_dovish_model")
tokenizer.save_pretrained("MODELS_II/electra_hawkish_dovish_model")


In [ ]:

from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# 🔹 Tahminleri al
preds = trainer.predict(tokenized["test"])
y_true = np.array(tokenized["test"]["label"])
y_pred = np.argmax(preds.predictions, axis=1)

# 🔹 Sınıf isimleri
label_names = ["neutral", "hawkish", "dovish"]

# 🔹 Genel sınıflandırma raporu
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

# 🔹 Genel accuracy
overall_acc = accuracy_score(y_true, y_pred)
print(f"✅ Overall Accuracy: {overall_acc:.4f}")

# 🔹 Her sınıf için accuracy (class-level)
for label_id, label_name in enumerate(label_names):
    class_mask = (y_true == label_id)
    class_acc = accuracy_score(y_true[class_mask], y_pred[class_mask])
    print(f"🔸 Accuracy for class '{label_name}': {class_acc:.4f}")

# ALBERT

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 🔹 1. Veri Yükleme
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 2. Label Mapping
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 3. Hugging Face Dataset'e Dönüştür
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 4. Tokenizer ve Model (ALBERT)
model_name = "albert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 5. Tokenization
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 6. Değerlendirme Fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 7. Eğitim Ayarları
training_args = TrainingArguments(
    output_dir="./albert_hawkish_dovish_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    num_train_epochs=6,
    learning_rate=1.8075447749523194e-05,
    weight_decay=0.1701694123118962,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 🔹 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 🔹 9. Eğitimi Başlat
trainer.train()

# 🔹 10. Modeli Kaydet
trainer.save_model("MODELS_II/albert_hawkish_dovish_model")
tokenizer.save_pretrained("MODELS_II/albert_hawkish_dovish_model")


In [ ]:

from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# 🔹 Tahminleri al
preds = trainer.predict(tokenized["test"])
y_true = np.array(tokenized["test"]["label"])
y_pred = np.argmax(preds.predictions, axis=1)

# 🔹 Sınıf isimleri
label_names = ["neutral", "hawkish", "dovish"]

# 🔹 Genel sınıflandırma raporu
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

# 🔹 Genel accuracy
overall_acc = accuracy_score(y_true, y_pred)
print(f"✅ Overall Accuracy: {overall_acc:.4f}")

# 🔹 Her sınıf için accuracy (class-level)
for label_id, label_name in enumerate(label_names):
    class_mask = (y_true == label_id)
    class_acc = accuracy_score(y_true[class_mask], y_pred[class_mask])
    print(f"🔸 Accuracy for class '{label_name}': {class_acc:.4f}")

# LONGFORMER

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 🔹 1. CSV'den veri yükle
df = pd.read_csv("BALANCED_DATASET.csv")

# 🔹 2. Label mapping
label_map = {0: "neutral", 1: "hawkish", 2: "dovish"}
id2label = {v: k for k, v in label_map.items()}

# 🔹 3. Hugging Face Dataset'e dönüştür
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

# 🔹 4. Tokenizer & Model (Longformer)
model_name = "allenai/longformer-base-4096"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 🔹 5. Tokenization (uzun girişlere uygun padding ve truncation ayarı!)
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=4096)

tokenized = dataset.map(tokenize_fn, batched=True)

# 🔹 6. Değerlendirme metrikleri
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 🔹 7. Eğitim ayarları
training_args = TrainingArguments(
    output_dir="./longformer_hawkish_dovish_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=4,  # VRAM için düşürüldü
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

# 🔹 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# # 🔹 9. Eğitimi başlat
# trainer.train()

# # 🔹 10. Modeli kaydet
# trainer.save_model("./longformer_hawkish_dovish_model")
# tokenizer.save_pretrained("./longformer_hawkish_dovish_model")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

preds = trainer.predict(tokenized["test"])
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
print(classification_report(y_true, y_pred, target_names=["neutral", "hawkish", "dovish"]))
y_true = tokenized["test"]["label"]
y_pred = np.argmax(preds.predictions, axis=1)
acc = accuracy_score(y_true, y_pred)
print(f"✅ Accuracy: {acc:.4f}")

# MACHINE LEARNING MODELS

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
import pandas as pd

In [ ]:
# CSV dosyasını yükle
df = pd.read_csv("BALANCED_DATASET.csv")

# Veri ve etiket
X = df["text"]
y = df["label"]  # Burada 0: neutral, 1: hawkish, 2: dovish

# Eğitim ve test bölmesi
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# TF-IDF vektörleştirici
vectorizer = TfidfVectorizer(max_features=5000)  # daha düşük boyutlu vektörlerle başlamak için

# Veriye uygula
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


## RANDOM FOREST

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Modeli eğitme ve tahmin
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_vec, y_train)
y_pred_rf = rf.predict(X_test_vec)

# Etiket isimleri
labels = ["neutral", "hawkish", "dovish"]

# Sınıflandırma raporu
print("🔍 Random Forest Classification Report")
report = classification_report(y_test, y_pred_rf, target_names=labels, digits=4)
print(report)

import numpy as np

print("🔢 Accuracy per class (based on correct predictions over total in class):")
for label in labels:
    # Pandas dizileriyle çalışmayı garanti altına al
    idx = y_test[y_test == label].index
    correct = np.sum(y_pred_rf[idx] == y_test[idx])
    total = len(idx)
    acc = correct / total if total > 0 else 0.0
    print(f"{label}: {acc:.4f}")


## SUPPORT VECTOR MACHINE

In [ ]:
svm = LinearSVC()
svm.fit(X_train_vec, y_train)
y_pred_svm = svm.predict(X_test_vec)

print("🔍 Support Vector Machine (LinearSVC) Classification Report")
print(classification_report(y_test, y_pred_svm, target_names=["neutral", "hawkish", "dovish"], digits=4))


## DECISION TREE

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_vec, y_train)
y_pred_dt = dt.predict(X_test_vec)

print("🔍 Decision Tree Classification Report")
print(classification_report(y_test, y_pred_dt, target_names=["neutral", "hawkish", "dovish"]))


## Multinomial Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)
y_pred_nb = nb.predict(X_test_vec)

print("🔍 Multinomial Naive Bayes Classification Report")
print(classification_report(y_test, y_pred_nb, target_names=["neutral", "hawkish", "dovish"]))


## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_vec, y_train)
y_pred_lr = lr.predict(X_test_vec)

print("🔍 Logistic Regression Classification Report")
print(classification_report(y_test, y_pred_lr, target_names=["neutral", "hawkish", "dovish"], digits = 4))


## Gradient Boosting

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")
xgb.fit(X_train_vec, y_train)
y_pred_xgb = xgb.predict(X_test_vec)

print("🔍 XGBoost Classification Report")
print(classification_report(y_test, y_pred_xgb, target_names=["neutral", "hawkish", "dovish"], digits = 4))


## K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_vec, y_train)
y_pred_knn = knn.predict(X_test_vec)

print("🔍 K-Nearest Neighbors Classification Report")
print(classification_report(y_test, y_pred_knn, target_names=["neutral", "hawkish", "dovish"]))
